In [4]:
# original code
# input Ui, Uj, Aij, Vi and g
# central averaging
# output: [dRi/dUi (5x5), dRi/dUj (5x5)]

from sympy import *

Ui = MatrixSymbol("U_i", 5, 1)    # conservative variables of cell i [rho, rhoux, rhouy, rhouz, rhoE]
Uj = MatrixSymbol("U_j", 5, 1)    #                      of cell j
Aij = MatrixSymbol("A_{ij}", 3, 1) # area normal vector [Ax, Ay, Az]
Vi = symbols("V_i")
g = symbols("gamma") # Explicitly define as symbol to avoid FunctionClass error

def dot(a, b):
    return Matrix(a.T @ b)[0]

# Convert to rimitive variables
rhoi = Ui[0]
ui = Ui[1:4] / rhoi
rhoj = Uj[0]
uj = Uj[1:4] / rhoj

# Ideal gas
pi = (Ui[4] - rhoi * dot(ui, ui) / 2) / (g - 1)
pj = (Uj[4] - rhoj * dot(uj, uj) / 2) / (g - 1)

# RHS (central) of i as a function of Ui and Uj
C = dot(rhoi * ui + rhoj * uj, Aij) / 2
M = Matrix((rhoi * ui @ ui.T + rhoj * uj @ uj.T).T @ Aij) / 2
G = Matrix((pi + pj) / 2 * Aij)
K = dot(((Ui[4] + pi) * ui + (Uj[4] + pj) * uj)/2, Aij)
Ri = -Matrix([[C], M + G, [K]]) / Vi

# Jacobian matrix [dRi/dUi (5x5), dRi/dUj (5x5)]
J = simplify(Ri.jacobian(Matrix([Ui, Uj])))
display(J)

# try lambdify to convert J into a function

Matrix([
[                                                                                                                                                                                                               0,                                                                                                                                                                                                                        -A_{ij}[0, 0]/(2*V_i),                                                                                                                                                                                                                        -A_{ij}[1, 0]/(2*V_i),                                                                                                                                                                                                                        -A_{ij}[2, 0]/(2*V_i),                                                                   

In [5]:
# modified version
# fluxes at cell boundary with Central Differencing (Ui + Uj)/2
massflux_avg = (rhoi * ui + rhoj * uj)/2
momentumflux_avg = (rhoi * ui @ ui.T + rhoj * uj @ uj.T)/2
pressureflux_avg = (pi + pj) / 2
energyflux_avg = ((Ui[4] + pi) * ui + (Uj[4] + pj) * uj)/2

# RHS (central) of i as a function of Ui and Uj
C = dot(massflux_avg, Aij)
M = Matrix(momentumflux_avg.T @ Aij)
G = Matrix(pressureflux_avg * Aij)
K = dot(energyflux_avg, Aij)
Ri = -Matrix([[C], M + G, [K]]) / Vi

# Jacobian matrix [dRi/dUi (5x5), dRi/dUj (5x5)]
J = simplify(Ri.jacobian(Matrix([Ui, Uj])))
display(J)

# try lambdify to convert J into a function

Matrix([
[                                                                                                                                                                                                               0,                                                                                                                                                                                                                        -A_{ij}[0, 0]/(2*V_i),                                                                                                                                                                                                                        -A_{ij}[1, 0]/(2*V_i),                                                                                                                                                                                                                        -A_{ij}[2, 0]/(2*V_i),                                                                   

In [6]:
# input Ui, Uj, Aij, Vi and g
# CHANGED TO Roe-averaged 
# output: [dRi/dUi (5x5), dRi/dUj (5x5)]

# --- ROE AVERAGING ---
# 1. Compute Roe-averaged variables
# ratio = rhoj/rhoi
R = sqrt(rhoj / rhoi)
# roe-averaged density
rho_roe = sqrt(rhoi * rhoj)

# roe-averaged velocities vector (3,1)
u_roe = (ui + R * uj) / (1 + R)

# compute total enthalpy : H
Hi = (Ui[4] + pi) / rhoi
Hj = (Uj[4] + pj) / rhoj

# roe-averaged total enthalpy
H_roe = (Hi + R * Hj) / (1 + R)

# Derive Roe-averaged pressure from enthalpy and velocity
# Formula: static pressure = (gamma - 1)/gamma * rho * (H - 0.5 * u^2)
# Roe-averaged static pressure
p_roe = (g - 1) / g * rho_roe * (H_roe - dot(u_roe, u_roe) / 2)

# 2. Construct the fluxes evaluated at the Roe-averaged state
massflux_roe = rho_roe * u_roe
momentumflux_roe = rho_roe * u_roe @ u_roe.T
pressureflux_roe = p_roe
energyflux_roe = rho_roe * H_roe * u_roe

# RHS of i as a function of Ui and Uj using Roe fluxes
# (Note: The /2 scaling is removed here so these represent the true boundary fluxes)
C = dot(massflux_roe, Aij)
M = Matrix(momentumflux_roe.T @ Aij)
G = Matrix(pressureflux_roe * Aij)
K = dot(energyflux_roe, Aij)

Ri = -Matrix([[C], M + G, [K]]) / Vi

# Jacobian matrix [dRi/dUi (5x5), dRi/dUj (5x5)]
J = simplify(Ri.jacobian(Matrix([Ui, Uj])))
display(J)

KeyboardInterrupt: 

In [7]:
# input Ui, Uj, Aij, Vi and g
# CHANGED TO Roe-averaged 
# output: roe-averaged matrix dR(tildeU)/tildeU
# directly use formula from chungs book
# return numerical values
# tildeU = roe-averaged of Ui and Uj
# roe-averaged matrix = (a1 * n1 + a2 * n2 + a3 * n3)* area = a1 * Aij[0] + a2 * Aij[1] + a3 * Aij[2]

# Assuming you have defined:
rho = Ui[0]
u, v, w = ui[0], ui[1], ui[2]
E = Ui[4]

# Helper terms to keep the matrix clean
q2 = u**2 + v**2 + w**2  # Velocity squared
g1 = g - 1
g3 = g - 3

# Matrix a1 (Flux Jacobian in x-direction)
a1 = Matrix([
    [0, 1, 0, 0, 0],
    [(g3/2)*u**2 + (g1/2)*(v**2 + w**2), (3-g)*u, (1-g)*v, (1-g)*w, g1],
    [-u*v, v, u, 0, 0],
    [-u*w, w, 0, u, 0],
    [-g*E*u + g1*u*q2, g*E + (1-g)/2*(3*u**2 + v**2 + w**2), (1-g)*u*v, (1-g)*u*w, g*u]
])

# Matrix a2 (Flux Jacobian in y-direction)
a2 = Matrix([
    [0, 0, 1, 0, 0],
    [-u*v, v, u, 0, 0],
    [(g3/2)*v**2 + (g1/2)*(u**2 + w**2), (1-g)*u, (3-g)*v, (1-g)*w, g1],
    [-v*w, 0, w, v, 0],
    [-g*E*v + g1*v*q2, (1-g)*u*v, g*E + (1-g)/2*(u**2 + 3*v**2 + w**2), (1-g)*v*w, g*v]
])

# Matrix a3 (Flux Jacobian in z-direction)
a3 = Matrix([
    [0, 0, 0, 1, 0],
    [-u*w, w, 0, u, 0],
    [-v*w, 0, w, v, 0],
    [(g3/2)*w**2 + (g1/2)*(u**2 + v**2), (1-g)*u, (1-g)*v, (3-g)*w, g1],
    [-g*E*w + g1*w*q2, (1-g)*u*w, (1-g)*v*w, g*E + (1-g)/2*(u**2 + v**2 + 3*w**2), g*w]
])


roe_averaged_jacobian = a1 * Aij[0] + a2 * Aij[1] + a3 * Aij[2]

display(roe_averaged_jacobian)
# lamdify the function later 
# the inputs are roe-averaged quantities
# output the roe-averaged jacobian matrix directly (with numerical values)

Matrix([
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                0,                                                                                                                                                                                                                                                   A_{ij}[0, 0],                                                                                                                                                                                                                            

In [8]:
# viscous part 
def compute_viscous_spectral_radius(mu, Pr, gamma, rho, xi, xj, mu_t=0.0, Pr_t=0.9):
    """
    Compute the viscous flux Jacobian approximation (spectral radius form).
    
    Based on equation (2.62) from Roger's
    |λ_max_ij| = (1/l_ij) * max(4/(3*rho_ij), gamma/rho_ij) * (mu*l_ij/Pr_l + mu_t_ij/Pr_t)
    
    Parameters
    ----------
    mu    : float  - dynamic viscosity (laminar), μ
    Pr    : float  - laminar Prandtl number, Pr_l
    gamma : float  - specific heat coefficient, γ
    rho   : float  - density, ρ_ij (at the edge midpoint or averaged)
    xi    : array-like (2D or 3D) - position vector of node i
    xj    : array-like (2D or 3D) - position vector of node j
    mu_t  : float  - turbulent dynamic viscosity, μ_t (default 0, laminar)
    Pr_t  : float  - turbulent Prandtl number, Pr_t (default 0.9)
    
    Returns
    -------
    lambda_max : float  - spectral radius |λ_max_ij|
    J_G        : ndarray - approximated Jacobian matrix (scalar * Identity)
    """
    xi = np.asarray(xi, dtype=float)
    xj = np.asarray(xj, dtype=float)

    # Edge vector length: l_ij = |x_j - x_i|
    l_ij = np.linalg.norm(xj - xi)
    if l_ij == 0.0:
        raise ValueError("Nodes xi and xj are coincident (l_ij = 0).")

    # Viscous scaling terms
    term1 = 4.0 / (3.0 * rho)   # 4 / (3 * rho_ij)
    term2 = gamma / rho          # gamma / rho_ij

    max_term = max(term1, term2)

    # Prandtl-weighted viscosity sum
    prandtl_term = (mu / Pr) + (mu_t / Pr_t)

    # Spectral radius (eq. 2.62)
    lambda_max = (1.0 / l_ij) * max_term * prandtl_term

    # Approximated Jacobian J_G = |lambda_max| * I  (eq. 2.61)
    n = len(xi)
    J_G = lambda_max * np.eye(n)

    return lambda_max, J_G

In [ ]:
# inviscid part by central-averaging of Ui and Uj
from sympy import *

def build_inviscid_jacobian():
    # ── Symbolic variables ────────────────────────────────────────────────────
    Ui  = Matrix(list(symbols("ui0:5")))   # shape (5,1) — now supports / and slicing
    Uj  = Matrix(list(symbols("uj0:5")))
    Ax, Ay, Az = symbols("Ax Ay Az")
    Aij = Matrix([Ax, Ay, Az])
    Vi  = symbols("V_i")
    g   = symbols("gamma")

    def dot(a, b):
        return (a.T @ b)[0, 0]

    # ── Primitive variables ───────────────────────────────────────────────────
    rhoi = Ui[0]
    ui   = Ui[1:4, :] / rhoi          # (3,1) Matrix / Symbol ✓
    rhoj = Uj[0]
    uj   = Uj[1:4, :] / rhoj

    # ── Ideal gas pressure ────────────────────────────────────────────────────
    pi = (Ui[4] - rhoi * dot(ui, ui) / 2) * (g - 1)
    pj = (Uj[4] - rhoj * dot(uj, uj) / 2) * (g - 1)

    # ── Central flux RHS ─────────────────────────────────────────────────────
    C = dot(rhoi * ui + rhoj * uj, Aij) / 2
    M = (rhoi * ui * ui.T + rhoj * uj * uj.T) @ Aij / 2
    G = (pi + pj) / 2 * Aij
    K = dot(((Ui[4] + pi) * ui + (Uj[4] + pj) * uj) / 2, Aij)

    Ri = -Matrix([[C], M + G, [K]]) / Vi

    # ── Symbolic Jacobian ─────────────────────────────────────────────────────
    print("Computing symbolic Jacobian (this may take a moment)...")
    all_vars    = Matrix([*Ui, *Uj])
    J_sym       = simplify(Ri.jacobian(all_vars))

    dRi_dUi_sym = J_sym[:, :5]
    dRi_dUj_sym = J_sym[:, 5:]

    # ── Lambdify ──────────────────────────────────────────────────────────────
    sym_args = (*Ui, *Uj, Ax, Ay, Az, Vi, g)   # 15 scalar arguments

    dRi_dUi_fn = lambdify(sym_args, dRi_dUi_sym, modules="numpy")
    dRi_dUj_fn = lambdify(sym_args, dRi_dUj_sym, modules="numpy")

    def jacobian_fn(Ui_val, Uj_val, Aij_val, Vi_val, gamma_val):
        import numpy as np
        Ui_val  = np.asarray(Ui_val,  dtype=float)
        Uj_val  = np.asarray(Uj_val,  dtype=float)
        Aij_val = np.asarray(Aij_val, dtype=float)

        args = (*Ui_val, *Uj_val, *Aij_val, float(Vi_val), float(gamma_val))

        dRi_dUi = np.array(dRi_dUi_fn(*args), dtype=float)
        dRi_dUj = np.array(dRi_dUj_fn(*args), dtype=float)

        return dRi_dUi, dRi_dUj

    return jacobian_fn


# ── Build once ────────────────────────────────────────────────────────────────
compute_inviscid_jacobian = build_inviscid_jacobian()


# ── Example usage ─────────────────────────────────────────────────────────────
if __name__ == "__main__":
    import numpy as np

    gamma = 1.4
    rho_i, rho_j = 1.0, 1.1

    Ui = np.array([rho_i, rho_i*0.5, rho_i*0.2, 0.0, rho_i*2.5])
    Uj = np.array([rho_j, rho_j*0.4, rho_j*0.1, 0.0, rho_j*2.3])

    Aij   = np.array([1.0, 0.0, 0.0])
    Vi    = 0.1

    dRi_dUi, dRi_dUj = compute_inviscid_jacobian(Ui, Uj, Aij, Vi, gamma)

    np.set_printoptions(precision=4, suppress=True, linewidth=120)
    print("\ndRi/dUi (5x5):\n", dRi_dUi)
    print("\ndRi/dUj (5x5):\n", dRi_dUj)

Computing symbolic Jacobian (this may take a moment)...

dRi/dUi (5x5):
 [[  0.    -5.    -0.    -0.     0.  ]
 [  0.96  -4.     0.4    0.    -2.  ]
 [  0.5   -1.    -2.5    0.    -0.  ]
 [  0.     0.     0.    -2.5   -0.  ]
 [  8.46 -16.71   0.2    0.    -3.5 ]]

dRi/dUj (5x5):
 [[  0.     -5.     -0.     -0.      0.   ]
 [  0.63   -3.2     0.2     0.     -2.   ]
 [  0.2    -0.5    -2.      0.     -0.   ]
 [  0.      0.      0.     -2.     -0.   ]
 [  6.304 -15.61    0.08    0.     -2.8  ]]


In [11]:
# roe-averaged inviscid jacobian
def build_inviscid_jacobian_roe():
    # ── Symbolic variables ────────────────────────────────────────────────────
    Ui  = Matrix(list(symbols("ui0:5")))
    Uj  = Matrix(list(symbols("uj0:5")))
    Ax, Ay, Az = symbols("Ax Ay Az")
    Aij = Matrix([Ax, Ay, Az])
    Vi  = symbols("V_i")
    g   = symbols("gamma")

    def dot(a, b):
        return (a.T @ b)[0, 0]

    # ── Primitive variables ───────────────────────────────────────────────────
    rhoi = Ui[0]
    ui   = Ui[1:4, :] / rhoi
    rhoj = Uj[0]
    uj   = Uj[1:4, :] / rhoj

    # ── Ideal gas pressure ────────────────────────────────────────────────────
    pi = (Ui[4] - rhoi * dot(ui, ui) / 2) * (g - 1)
    pj = (Uj[4] - rhoj * dot(uj, uj) / 2) * (g - 1)

    # ── Roe averaging ─────────────────────────────────────────────────────────
    R       = sqrt(rhoj / rhoi)
    rho_roe = sqrt(rhoi * rhoj)
    u_roe   = (ui + R * uj) / (1 + R)

    Hi      = (Ui[4] + pi) / rhoi
    Hj      = (Uj[4] + pj) / rhoj
    H_roe   = (Hi + R * Hj) / (1 + R)

    p_roe   = (g - 1) / g * rho_roe * (H_roe - dot(u_roe, u_roe) / 2)

    # ── Roe fluxes ────────────────────────────────────────────────────────────
    massflux_roe     = rho_roe * u_roe
    momentumflux_roe = rho_roe * u_roe * u_roe.T
    energyflux_roe   = rho_roe * H_roe * u_roe

    C = dot(massflux_roe, Aij)
    M = Matrix(momentumflux_roe.T @ Aij)
    G = p_roe * Aij
    K = dot(energyflux_roe, Aij)

    Ri = -Matrix([[C], M + G, [K]]) / Vi

    # ── Symbolic Jacobian ─────────────────────────────────────────────────────
    print("Computing Roe Jacobian symbolically (may take a while)...")
    all_vars    = Matrix([*Ui, *Uj])
    J_sym       = simplify(Ri.jacobian(all_vars))

    dRi_dUi_sym = J_sym[:, :5]
    dRi_dUj_sym = J_sym[:, 5:]

    # ── Lambdify ──────────────────────────────────────────────────────────────
    sym_args = (*Ui, *Uj, Ax, Ay, Az, Vi, g)

    dRi_dUi_fn = lambdify(sym_args, dRi_dUi_sym, modules="numpy")
    dRi_dUj_fn = lambdify(sym_args, dRi_dUj_sym, modules="numpy")

    def jacobian_fn(Ui_val, Uj_val, Aij_val, Vi_val, gamma_val):
        """
        Evaluate the Roe-averaged inviscid flux Jacobian numerically.

        Parameters
        ----------
        Ui_val    : array-like, shape (5,)  conservative vars of cell i  [rho, rho*ux, rho*uy, rho*uz, rho*E]
        Uj_val    : array-like, shape (5,)  conservative vars of cell j
        Aij_val   : array-like, shape (3,)  area-weighted face normal [Ax, Ay, Az]
        Vi_val    : float                   volume of cell i
        gamma_val : float                   ratio of specific heats

        Returns
        -------
        dRi_dUi : np.ndarray, shape (5, 5)   d(Ri)/d(Ui)
        dRi_dUj : np.ndarray, shape (5, 5)   d(Ri)/d(Uj)
        """
        import numpy as np
        Ui_val  = np.asarray(Ui_val,  dtype=float)
        Uj_val  = np.asarray(Uj_val,  dtype=float)
        Aij_val = np.asarray(Aij_val, dtype=float)

        args = (*Ui_val, *Uj_val, *Aij_val, float(Vi_val), float(gamma_val))

        dRi_dUi = np.array(dRi_dUi_fn(*args), dtype=float)
        dRi_dUj = np.array(dRi_dUj_fn(*args), dtype=float)

        return dRi_dUi, dRi_dUj

    return jacobian_fn


# ── Build once ────────────────────────────────────────────────────────────────
compute_inviscid_jacobian_roe = build_inviscid_jacobian_roe()


# ── Example usage ─────────────────────────────────────────────────────────────
if __name__ == "__main__":
    import numpy as np

    gamma = 1.4
    rho_i, rho_j = 1.0, 1.1

    Ui = np.array([rho_i, rho_i*0.5, rho_i*0.2, 0.0, rho_i*2.5])
    Uj = np.array([rho_j, rho_j*0.4, rho_j*0.1, 0.0, rho_j*2.3])

    Aij = np.array([1.0, 0.0, 0.0])
    Vi  = 0.1

    dRi_dUi, dRi_dUj = compute_inviscid_jacobian_roe(Ui, Uj, Aij, Vi, gamma)

    np.set_printoptions(precision=4, suppress=True, linewidth=120)
    print("\ndRi/dUi (5x5):\n", dRi_dUi)
    print("\ndRi/dUj (5x5):\n", dRi_dUj)

    # ── Sanity check: columns should sum close to zero (flux consistency) ─────
    print("\nColumn sums of [dRi_dUi | dRi_dUj] (expect ~0 for uniform flow):")
    print(np.round(dRi_dUi.sum(axis=0), 6))
    print(np.round(dRi_dUj.sum(axis=0), 6))

Computing Roe Jacobian symbolically (may take a while)...

dRi/dUi (5x5):
 [[  0.075   -5.1191  -0.      -0.       0.    ]
 [  0.8345  -3.6461   0.3347   0.      -2.0476]
 [  0.4118  -0.7618  -2.2975   0.      -0.    ]
 [  0.       0.       0.      -2.2975  -0.    ]
 [  7.8724 -16.4896   0.1838   0.      -3.2165]]

dRi/dUj (5x5):
 [[ -0.0681  -4.8809  -0.      -0.       0.    ]
 [  0.7328  -3.5322   0.2633   0.      -1.9524]
 [  0.2624  -0.7263  -2.1906   0.      -0.    ]
 [  0.       0.       0.      -2.1906  -0.    ]
 [  6.816  -15.8099   0.0876   0.      -3.0668]]

Column sums of [dRi_dUi | dRi_dUj] (expect ~0 for uniform flow):
[  9.1937 -26.0166  -1.779   -2.2975  -5.2642]
[  7.743  -24.9492  -1.8397  -2.1906  -5.0192]
